<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/fftres.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import glob
import copy
import random
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet34, ResNet34_Weights
import albumentations as A

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 8
EPOCHS = 100
LR = 1e-4
IMG_SIZE = 256
SEED = 42
N_SPLITS = 5
TEST_SIZE = 0.15
CLASSES = ["benign", "malignant"]
SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

class BUSIDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None, indices=None):
        self.transform = transform
        self._all_samples = []
        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir): continue
            images = sorted([f for f in os.listdir(cls_dir) if f.endswith(".png") and "_mask" not in f])
            label = 0 if cls == "benign" else 1
            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = sorted([f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask") and f.endswith(".png")])
                if not mask_files: continue
                self._all_samples.append((img_path, [os.path.join(cls_dir, f) for f in mask_files], label))
        if indices is not None:
            self._all_samples = [self._all_samples[i] for i in indices]

    def __len__(self): return len(self._all_samples)

    def __getitem__(self, idx):
        img_path, mask_paths, _ = self._all_samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            m = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, m > 0).astype(np.uint8)
        combined_mask = combined_mask.astype(np.float32)
        if self.transform:
            aug = self.transform(image=image, mask=combined_mask)
            image, combined_mask = aug["image"], aug["mask"]
        combined_mask = (combined_mask > 0.5).astype(np.float32)
        return torch.from_numpy(image).permute(2, 0, 1).float(), torch.from_numpy(combined_mask).unsqueeze(0).float()

    @property
    def labels(self): return [s[2] for s in self._all_samples]

class Haar2D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.in_channels = in_channels
        ll = torch.tensor([[0.5, 0.5], [0.5, 0.5]])
        lh = torch.tensor([[-0.5, -0.5], [0.5, 0.5]])
        hl = torch.tensor([[-0.5, 0.5], [-0.5, 0.5]])
        hh = torch.tensor([[0.5, -0.5], [-0.5, 0.5]])
        filters = torch.stack([ll, lh, hl, hh]).unsqueeze(1)
        self.register_buffer("filters", filters.repeat(in_channels, 1, 1, 1))

    def forward(self, x):
        return F.conv2d(x, self.filters, stride=2, groups=self.in_channels)

class WaveletEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.dwt1 = Haar2D(3)
        self.conv1 = nn.Conv2d(12, 64, 3, padding=1)
        self.dwt2 = Haar2D(64)
        self.conv2 = nn.Conv2d(256, 64, 3, padding=1)
        self.dwt3 = Haar2D(64)
        self.conv3 = nn.Conv2d(256, 128, 3, padding=1)
        self.dwt4 = Haar2D(128)
        self.conv4 = nn.Conv2d(512, 256, 3, padding=1)
        self.dwt5 = Haar2D(256)
        self.conv5 = nn.Conv2d(1024, 512, 3, padding=1)

    def forward(self, x):
        w0 = F.relu(self.conv1(self.dwt1(x)))
        w1 = F.relu(self.conv2(self.dwt2(w0)))
        w2 = F.relu(self.conv3(self.dwt3(w1)))
        w3 = F.relu(self.conv4(self.dwt4(w2)))
        w4 = F.relu(self.conv5(self.dwt5(w3)))
        return w0, w1, w2, w3, w4

class PCBAM_Filter(nn.Module):
    def __init__(self, in_channels, reduction=8):
        super().__init__()
        self.cam = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels)
        )
        self.sam = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.cam(F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        mxp = self.cam(F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        x_c = x * torch.sigmoid(avg + mxp)
        sp = torch.cat([torch.mean(x_c, dim=1, keepdim=True), torch.max(x_c, dim=1, keepdim=True)[0]], dim=1)
        return x_c * torch.sigmoid(self.sam(sp))

class SpatialGateAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        mid = in_channels // 8
        self.q_conv = nn.Conv2d(in_channels, mid, 1)
        self.k_conv = nn.Conv2d(in_channels, mid, 1)
        self.v_conv = nn.Conv2d(in_channels, in_channels, 1)
        self.gate = nn.Conv2d(mid * 2 + in_channels, in_channels, 1)

    def forward(self, x):
        q, k, v = self.q_conv(x), self.k_conv(x), self.v_conv(x)
        A = torch.sigmoid(self.gate(torch.cat([q, k, v], dim=1)))
        return x + v * A

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)

class DualPathWaveletUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.wavelet_encoder = WaveletEncoder()
        resnet = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        self.conv1, self.bn1, self.relu, self.maxpool = resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4

        self.fuse0 = nn.Conv2d(64 + 64, 64, 1)
        self.fuse1 = nn.Conv2d(64 + 64, 64, 1)
        self.fuse2 = nn.Conv2d(128 + 128, 128, 1)
        self.fuse3 = nn.Conv2d(256 + 256, 256, 1)
        self.fuse4 = nn.Conv2d(512 + 512, 512, 1)

        self.pcbam1 = PCBAM_Filter(64)
        self.pcbam2 = PCBAM_Filter(128)
        self.pcbam3 = PCBAM_Filter(256)
        self.pcbam4 = PCBAM_Filter(512)

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.sga = SpatialGateAttention(1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.up0 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec0 = DoubleConv(128, 64)
        self.up_out = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec_out = DoubleConv(32, 32)
        self.final = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        w0, w1, w2, w3, w4 = self.wavelet_encoder(x)
        x0 = self.relu(self.bn1(self.conv1(x)))
        f0 = self.fuse0(torch.cat([x0, w0], dim=1))

        e1 = self.layer1(self.maxpool(x0))
        f1 = self.fuse1(torch.cat([e1, w1], dim=1))

        e2 = self.layer2(e1)
        f2 = self.fuse2(torch.cat([e2, w2], dim=1))

        e3 = self.layer3(e2)
        f3 = self.fuse3(torch.cat([e3, w3], dim=1))

        e4 = self.layer4(e3)
        f4 = self.fuse4(torch.cat([e4, w4], dim=1))

        s1, s2, s3, s4 = self.pcbam1(f1), self.pcbam2(f2), self.pcbam3(f3), self.pcbam4(f4)
        b = self.sga(self.bottleneck(self.pool(f4)))

        d4 = self.dec4(torch.cat([self.up4(b), s4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))
        d0 = self.dec0(torch.cat([self.up0(d1), f0], dim=1))
        out = self.dec_out(self.up_out(d0))

        return self.final(out)

class HybridLoss(nn.Module):
    def __init__(self, smooth=1e-5, focal_gamma=2.0):
        super().__init__()
        self.smooth = smooth
        self.focal_gamma = focal_gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        probs, tgt = torch.sigmoid(logits).view(-1), targets.view(-1)
        inter = (probs * tgt).sum()
        dice = 1.0 - (2.0 * inter + self.smooth) / (probs.sum() + tgt.sum() + self.smooth)
        pt = torch.where(tgt == 1, probs, 1.0 - probs)
        focal = (-(1.0 - pt) ** self.focal_gamma * torch.log(pt + 1e-8)).mean()
        return 0.4 * bce + 0.4 * dice + 0.2 * focal

def compute_metrics(logits, targets, threshold=0.5, smooth=1e-5):
    preds, tgts = (torch.sigmoid(logits) > threshold).float().view(-1), targets.view(-1).float()
    tp, fp, fn = (preds * tgts).sum(), (preds * (1 - tgts)).sum(), ((1 - preds) * tgts).sum()
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    return {
        "dice": ((2 * tp + smooth) / (2 * tp + fp + fn + smooth)).item(),
        "iou": ((tp + smooth) / (tp + fp + fn + smooth)).item(),
        "precision": precision.item(),
        "recall": recall.item(),
        "f1": ((2 * precision * recall) / (precision + recall + smooth)).item()
    }

def aggregate_metrics(metric_list):
    return {k: float(np.mean([m[k] for m in metric_list])) for k in metric_list[0].keys()}

def predict_tta(model, image_tensor):
    with torch.amp.autocast("cuda"):
        p_orig = torch.sigmoid(model(image_tensor))
        p_hflip = torch.flip(torch.sigmoid(model(torch.flip(image_tensor, dims=[3]))), dims=[3])
        p_vflip = torch.flip(torch.sigmoid(model(torch.flip(image_tensor, dims=[2]))), dims=[2])
    return (p_orig + p_hflip + p_vflip) / 3.0

possible_paths = glob.glob("/kaggle/input/**/benign", recursive=True)
if not possible_paths: raise FileNotFoundError("Dataset not found. Check Kaggle input path.")
BASE_DIR = os.path.dirname(possible_paths[0])
full_dataset = BUSIDataset(BASE_DIR, classes=CLASSES, transform=None)
all_indices, all_labels = np.arange(len(full_dataset)), np.array(full_dataset.labels)

trainval_idx, test_idx = train_test_split(all_indices, test_size=TEST_SIZE, stratify=all_labels, random_state=SEED)
trainval_labels = all_labels[trainval_idx]
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
criterion = HybridLoss()
fold_paths, fold_val_metrics = [], []

for fold, (rel_train_idx, rel_val_idx) in enumerate(skf.split(trainval_idx, trainval_labels)):
    train_loader = DataLoader(BUSIDataset(BASE_DIR, CLASSES, train_transform, indices=trainval_idx[rel_train_idx]), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(BUSIDataset(BASE_DIR, CLASSES, val_test_transform, indices=trainval_idx[rel_val_idx]), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = DualPathWaveletUNet().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    best_val_dice = 0.0

    for epoch in range(EPOCHS):
        model.train()
        for images, masks in tqdm(train_loader, desc=f"F{fold+1} E{epoch+1:03d}", leave=False):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterion(model(images), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        model.eval()
        val_metric_list = []
        with torch.no_grad():
            for images, masks in val_loader:
                with torch.amp.autocast("cuda"):
                    val_metric_list.append(compute_metrics(model(images.to(device)), masks.to(device)))
        avg_val_dice = aggregate_metrics(val_metric_list)["dice"]

        if avg_val_dice > best_val_dice:
            best_val_dice = avg_val_dice
            save_path = os.path.join(SAVE_DIR, f"best_model_fold_{fold+1}.pth")
            torch.save(model.state_dict(), save_path)

    fold_paths.append(save_path)
    fold_val_metrics.append(best_val_dice)

test_loader = DataLoader(BUSIDataset(BASE_DIR, CLASSES, val_test_transform, indices=test_idx), batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
ensemble = []
for path in fold_paths:
    m = DualPathWaveletUNet().to(device)
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    ensemble.append(m)

configs = {
    "Single model (fold 1), no TTA": (ensemble[:1], False),
    "Ensemble (5 models), no TTA": (ensemble, False),
    "Ensemble (5 models) + TTA": (ensemble, True),
}

results_table = {}
for config_name, (models, use_tta) in configs.items():
    all_metrics = []
    with torch.no_grad():
        for images, masks in test_loader:
            images, masks = images.to(device), masks.to(device)
            avg_probs = torch.zeros_like(masks).to(device)
            for m in models:
                if use_tta: avg_probs += predict_tta(m, images)
                else:
                    with torch.amp.autocast("cuda"): avg_probs += torch.sigmoid(m(images))
            avg_probs /= len(models)
            pseudo_logit = torch.log(avg_probs.clamp(1e-6, 1 - 1e-6) / (1 - avg_probs.clamp(1e-6, 1 - 1e-6)))
            all_metrics.append(compute_metrics(pseudo_logit, masks))
    results_table[config_name] = aggregate_metrics(all_metrics)

print("\n ABLATION TABLE")
print("=" * 80)
print(f"{'Configuration':<40} {'Dice':>6} {'IoU':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
print("-" * 80)
for name, m in results_table.items():
    print(f"{name:<40} {m['dice']:>6.4f} {m['iou']:>6.4f} {m['precision']:>6.4f} {m['recall']:>6.4f} {m['f1']:>6.4f}")

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 216MB/s]



 ABLATION TABLE
Configuration                              Dice    IoU   Prec    Rec     F1
--------------------------------------------------------------------------------
Single model (fold 1), no TTA            0.7857 0.6756 0.8317 0.7656 0.7857
Ensemble (5 models), no TTA              0.8403 0.7356 0.8631 0.8321 0.8403
Ensemble (5 models) + TTA                0.8418 0.7375 0.8572 0.8409 0.8418
